# Diabetes Prediction — Model Training

Train and evaluate 6 model configurations on the Pima Indians Diabetes dataset. All runs tracked with MLflow; best model selected by ROC-AUC.

In [ ]:
import sys
import os
import json
import pathlib
import warnings
import logging

# Make src/ importable from the notebooks/ directory
sys.path.insert(0, str(pathlib.Path('..').resolve()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve, ConfusionMatrixDisplay,
)
import xgboost as xgb
import mlflow
import mlflow.sklearn

# Suppress MLflow deprecation notices and pickle safety advisory
warnings.filterwarnings('ignore', category=FutureWarning, module='mlflow')
logging.getLogger('mlflow.models.model').setLevel(logging.ERROR)
logging.getLogger('mlflow.sklearn').setLevel(logging.ERROR)

from src.preprocess import clean_data, build_preprocessor, FEATURES, TARGET

plt.rcParams['figure.dpi'] = 100
print('All imports OK')

In [2]:
PROJECT_ROOT = pathlib.Path('..').resolve()

with open(PROJECT_ROOT / 'configs' / 'config.yaml') as f:
    config = yaml.safe_load(f)

DATA_PATH    = PROJECT_ROOT / config['data']['raw_path']
TEST_SIZE    = config['data']['test_size']
RAND_STATE   = config['data']['random_state']
EXPERIMENT  = config['mlflow']['experiment_name']
TRACK_URI   = str(PROJECT_ROOT / config['mlflow']['tracking_uri'])

print(f'Data:        {DATA_PATH}')
print(f'Test size:   {TEST_SIZE}')
print(f'Experiment:  {EXPERIMENT}')
print(f'MLflow URI:  {TRACK_URI}')

Data:        /Users/vishalkumar/Documents/Arushi_Docs/Triple_ten/TripleTenGitHub/diabetes_prediction/data/diabetes.csv
Test size:   0.2
Experiment:  diabetes_prediction
MLflow URI:  /Users/vishalkumar/Documents/Arushi_Docs/Triple_ten/TripleTenGitHub/diabetes_prediction/mlruns


## 1. Load and Clean Data

Replace sentinel zeros (biologically impossible) with `NaN`; drop rows missing Glucose or BMI.

In [3]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Raw shape: {df_raw.shape}')

df = clean_data(df_raw)
print(f'After cleaning: {df.shape}  ({len(df_raw) - len(df)} rows dropped)')

X = df[FEATURES]
y = df[TARGET]

print(f'\nClass balance (0=no diabetes, 1=diabetes):')
print(y.value_counts(normalize=True).round(3).to_string())

Raw shape: (768, 9)
After cleaning: (752, 9)  (16 rows dropped)

Class balance (0=no diabetes, 1=diabetes):
Outcome
0    0.649
1    0.351


## 2. Train / Test Split

Stratified 80/20 split ensures both sets preserve the ~35% positive-class rate.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RAND_STATE,
    stratify=y,
)

print(f'Train: {X_train.shape[0]} rows | positive rate: {y_train.mean():.1%}')
print(f'Test:  {X_test.shape[0]} rows | positive rate: {y_test.mean():.1%}')

Train: 601 rows | positive rate: 35.1%
Test:  151 rows | positive rate: 35.1%


## 3. MLflow Setup and Training Helper

Each model runs inside a `Pipeline` (imputer → scaler → classifier) to prevent data leakage. All params, metrics, and the model artifact are logged per run.

In [ ]:
mlflow.set_tracking_uri(TRACK_URI)
mlflow.set_experiment(EXPERIMENT)

def evaluate_and_log(run_name, classifier, params):
    """Build pipeline, train, compute 5 metrics, log everything to MLflow."""
    pipeline = Pipeline([
        ('preprocessor', build_preprocessor()),
        ('classifier', classifier),
    ])
    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tag('model_type', run_name)

        # Log data version / description alongside model hyperparameters
        mlflow.set_tag('data_description',
            f'Pima Indians Diabetes | {len(X_train)} train / {len(X_test)} test | '
            f'sentinel zeros replaced with NaN | rows missing Glucose or BMI dropped')
        mlflow.log_param('train_rows', len(X_train))
        mlflow.log_param('test_rows', len(X_test))
        mlflow.log_param('n_features', len(FEATURES))
        mlflow.log_param('imputation', 'median')
        mlflow.log_param('scaling', 'standard_scaler')

        mlflow.log_params(params)

        pipeline.fit(X_train, y_train)

        y_pred = pipeline.predict(X_test)
        y_prob = pipeline.predict_proba(X_test)[:, 1]

        metrics = {
            'accuracy':  round(accuracy_score(y_test, y_pred),  4),
            'precision': round(precision_score(y_test, y_pred), 4),
            'recall':    round(recall_score(y_test, y_pred),    4),
            'f1':        round(f1_score(y_test, y_pred),        4),
            'roc_auc':   round(roc_auc_score(y_test, y_prob),  4),
        }
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(pipeline, name='model')

        run_id = run.info.run_id
        print(f'--- {run_name} ---')
        for k, v in metrics.items():
            print(f'  {k:<12}: {v}')
        print(f'  run_id      : {run_id}')
        return run_id, metrics, pipeline

results   = {}   # {name: {'run_id': ..., 'metrics': {...}}}
pipelines = {}   # {name: fitted sklearn Pipeline}
print('MLflow experiment ready:', EXPERIMENT)

## 4. Model 1 — Logistic Regression

Linear baseline with L2 regularization.

In [6]:
lr_params = config['models']['logistic_regression']

run_id, metrics, pipe = evaluate_and_log(
    'logistic_regression',
    LogisticRegression(**lr_params),
    lr_params,
)
results['logistic_regression']   = {'run_id': run_id, 'metrics': metrics}
pipelines['logistic_regression'] = pipe

2026/05/11 20:19:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/11 20:19:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- logistic_regression ---
  accuracy    : 0.755
  precision   : 0.7857
  recall      : 0.4151
  f1          : 0.5432
  roc_auc     : 0.85
  run_id      : af80658b4bb144b9a180ea6874be7639


## 5. Model 2 — Random Forest

Ensemble of decision trees — handles non-linear relationships and is robust to outliers.

In [7]:
rf_params = config['models']['random_forest']

run_id, metrics, pipe = evaluate_and_log(
    'random_forest',
    RandomForestClassifier(**rf_params),
    rf_params,
)
results['random_forest']   = {'run_id': run_id, 'metrics': metrics}
pipelines['random_forest'] = pipe

2026/05/11 20:19:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/11 20:19:57 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- random_forest ---
  accuracy    : 0.7616
  precision   : 0.7931
  recall      : 0.434
  f1          : 0.561
  roc_auc     : 0.8512
  run_id      : 9d0debf80a56455796eea397ce448b87


## 6. Model 3 — XGBoost

Gradient boosting — typically the strongest performer on tabular datasets.

In [8]:
xgb_params = config['models']['xgboost']

run_id, metrics, pipe = evaluate_and_log(
    'xgboost',
    xgb.XGBClassifier(**xgb_params, eval_metric='logloss', verbosity=0),
    xgb_params,
)
results['xgboost']   = {'run_id': run_id, 'metrics': metrics}
pipelines['xgboost'] = pipe

2026/05/11 20:19:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


2026/05/11 20:19:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- xgboost ---
  accuracy    : 0.755
  precision   : 0.7222
  recall      : 0.4906
  f1          : 0.5843
  roc_auc     : 0.8246
  run_id      : c416cc3d729a4759bcc5d18fdf29e708


## 7. Additional Experiment Variants

One variant per algorithm (6 runs total) to compare sensitivity to hyperparameter choices.

In [ ]:
# Logistic Regression v2: stronger L2 regularisation (C=0.01 vs default C=1.0)
lr_v2_params = {'C': 0.01, 'max_iter': 1000, 'solver': 'lbfgs'}

run_id, metrics, pipe = evaluate_and_log(
    'logistic_regression_v2',
    LogisticRegression(**lr_v2_params),
    lr_v2_params,
)
results['logistic_regression_v2']   = {'run_id': run_id, 'metrics': metrics}
pipelines['logistic_regression_v2'] = pipe

In [ ]:
# Random Forest v2: shallower trees + larger min_samples_leaf → higher bias, lower variance
rf_v2_params = {
    'n_estimators': 100, 'max_depth': 4, 'min_samples_leaf': 8, 'random_state': 42,
}

run_id, metrics, pipe = evaluate_and_log(
    'random_forest_v2',
    RandomForestClassifier(**rf_v2_params),
    rf_v2_params,
)
results['random_forest_v2']   = {'run_id': run_id, 'metrics': metrics}
pipelines['random_forest_v2'] = pipe

In [ ]:
# XGBoost v2: higher learning rate + fewer trees — tests faster convergence trade-off
xgb_v2_params = {
    'n_estimators': 100, 'learning_rate': 0.1, 'max_depth': 6,
    'subsample': 0.8, 'colsample_bytree': 0.8, 'random_state': 42,
}

run_id, metrics, pipe = evaluate_and_log(
    'xgboost_v2',
    xgb.XGBClassifier(**xgb_v2_params, eval_metric='logloss', verbosity=0),
    xgb_v2_params,
)
results['xgboost_v2']   = {'run_id': run_id, 'metrics': metrics}
pipelines['xgboost_v2'] = pipe

## 8. Compare All Results

In [9]:
rows = [{'model': name, **data['metrics']} for name, data in results.items()]
comparison = (
    pd.DataFrame(rows)
    .set_index('model')
    .sort_values('roc_auc', ascending=False)
)
print('Model comparison (sorted by ROC-AUC):')
display(comparison)

Model comparison (sorted by ROC-AUC):


,accuracy,precision,recall,f1,roc_auc
model,,,,,
random_forest,0.7616,0.7931,0.4340,0.5610,0.8512
logistic_regression,0.7550,0.7857,0.4151,0.5432,0.8500
xgboost,0.7550,0.7222,0.4906,0.5843,0.8246


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC-AUC bar chart
comparison['roc_auc'].plot(kind='barh', ax=axes[0],
                           color=[f'C{i}' for i in range(len(comparison))], alpha=0.85)
axes[0].set_xlim(0.5, 1.0)
axes[0].set_title('ROC-AUC by Model')
axes[0].axvline(0.8, color='gray', linestyle='--', alpha=0.5)

# ROC curves
for i, name in enumerate(comparison.index):
    y_prob = pipelines[name].predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    axes[1].plot(fpr, tpr, f'C{i}', label=f'{name} ({comparison.loc[name, "roc_auc"]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.4)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curves')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.show()

## 9. Programmatic Best-Run Selection with `mlflow.search_runs()`

Query the MLflow tracking server across all runs (including previous sessions) to identify the best run.

In [ ]:
all_runs = mlflow.search_runs(experiment_names=[EXPERIMENT], order_by=['metrics.roc_auc DESC'])

cols = ['tags.mlflow.runName', 'metrics.roc_auc', 'metrics.f1',
        'metrics.accuracy', 'metrics.recall', 'metrics.precision']
display(all_runs[cols].rename(columns=lambda c: c.split('.')[-1]).reset_index(drop=True))

best_row      = all_runs.iloc[0]
best_run_id   = best_row['run_id']
best_run_name = best_row['tags.mlflow.runName']
print(f'\nBest run: {best_run_name} | ROC-AUC: {best_row["metrics.roc_auc"]:.4f}')

## 10. Best Model Selection

Save the best run's metadata to `configs/best_model.json` for use in the LLM interface.

In [ ]:
best_name    = best_run_name
best_metrics = results[best_name]['metrics']
best_pipe    = pipelines[best_name]

print(f'Best model: {best_name}')
for k, v in best_metrics.items():
    print(f'  {k}: {v}')

best_info = {
    'model_name': best_name, 'run_id': best_run_id,
    'metrics': best_metrics, 'features': FEATURES,
    'tracking_uri': TRACK_URI, 'experiment_name': EXPERIMENT,
}
with open(PROJECT_ROOT / 'configs' / 'best_model.json', 'w') as f:
    json.dump(best_info, f, indent=2)
print('Saved to configs/best_model.json')

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test,
    best_pipe.predict(X_test),
    display_labels=['No Diabetes', 'Diabetes'],
    colorbar=False,
    ax=ax,
)
ax.set_title(f'Confusion Matrix — {best_name}')
plt.tight_layout()
plt.show()

## Summary

See `README.md` for the full preprocessing and model comparison summary.